# INFORME DE PROYECTO: BITÁCORA DE DESARROLLO Y SOLUCIÓN DE PROBLEMAS

**Proyecto:** Detector de Pose Fitness con MediaPipe y Gradio

**Alumno:** Christian Manuel Patricio Velasquez

**Fecha de Entrega:** 22 de junio de 2026

---

## 🛠️ Sesión 1: Corrección de Incompatibilidad en API de MediaPipe

**Fecha:** 19 de junio de 2026

### 1. Problema Inicial Detectado

Al ejecutar el notebook principal (`04_Proyecto_Pose_y_Despliegue.ipynb`), el sistema arrojaba el siguiente error crítico de detención:

`AttributeError: module 'mediapipe' has no attribute 'solutions'`.

* **Causa:** Incompatibilidad directa entre la versión de MediaPipe instalada en el entorno (v0.10.35) y el código base del repositorio, el cual utilizaba llamadas a la API obsoleta (`mp.solutions.pose`).



### 2. Análisis y Diagnóstico Técnico

Se descargó e inspeccionó la documentación oficial de Google (`[MediaPipe_Python_Tasks]_Pose_Landmarker`). Al contrastar las estructuras se identificaron 4 discrepancias críticas en el código heredado: el método de descarga del modelo, los módulos de importación, la clase del detector principal y las utilidades gráficas de dibujo manual con OpenCV.

### 3. Soluciones Implementadas

Se modificó la celda 7 del notebook reemplazando la arquitectura deprecada por la API moderna de tareas de visión (`mediapipe.tasks.python.vision`):

* **Modelo de detección:** Se programó la descarga automática desde el repositorio oficial de Google del archivo pesado `pose_landmarker_heavy.task` (~100 MB) ante su primera ejecución.


* **Inicialización y Dibujo:** Se migró a la clase oficial `vision.PoseLandmarker.create_from_options()` y se estandarizó el renderizado del esqueleto mediante `drawing_utils.draw_landmarks()`.


* **Resultado:** El script local compiló exitosamente, dejándose listo el archivo base `app.py` bajo un patrón de diseño limpio en 3 capas.



---

## 📈 Sesión 2: Finalización de Consignas y Robustez Postural

**Fecha:** 20 de junio de 2026 (Continuación)

### 1. Problema Observado

Los primeros testeos lógicos basados únicamente en la cadera derecha como dato aislado eran insuficientes para detectar anomalías complejas en gimnasios, tales como desequilibrios asimétricos, espaldas encorvadas o capturas erróneas con orientaciones invertidas.

### 2. Soluciones Implementadas (Desarrollo Lógico y Despliegue)

* **Consigna 1 (Métricas de Análisis):** Se completó la función `detectar_pose` incorporando 5 variables biomecánicas clave:


1. `distancia_hombros`: Control de encorvamiento de la zona superior (Rango óptimo: 0.08 - 0.15).


2. `visibilidad`: Confianza computacional del modelo sobre la captura.


3. `cadera_y` / *Desnivel pélvico*: Reemplazo de la cadera aislada por la altura promedio de ambas caderas para medir estabilidad.


4. `diferencia_rodillas`: Métrica propia para alertar desequilibrios en tren inferior.


5. `altura_cuerpo` e *Inclinación de Torso*: Monitoreo de extensión general y control ante inclinaciones lumbares extremas.




* *Filtro Antifraude:* Se codificó una regla explícita que compara la posición horizontal y vertical de la cabeza respecto a los tobillos, permitiendo vetar de forma automática capturas con posturas invertidas.




* **Consigna 2 (Arquitectura de Software):** Se estructuró la aplicación en 3 capas independientes (Data / Business Logic / Presentation con `gr.Blocks()`). Se eliminaron emojis en las cadenas de texto de respuesta para mitigar excepciones de codificación (`SyntaxError` por formato UTF-8) en terminales de sistemas operativos restrictivos.


* **Consigna 3 (Despliegue en Servidor):** Se subió la app a Hugging Face Spaces (`manuelcpv92/mi-app-gym`). El despliegue requirió resolver la autenticación por token personal debido a la baja del soporte de contraseñas de Git, solventar colisiones de ramas mediante `git pull origin main --rebase`, y forzar un archivo `packages.txt` con las librerías gráficas de bajo nivel (`libegl1` y `libgles2`) nativas del sistema operativo del contenedor (Debian Trixie).



---

## ⚙️ Sesión 3: Optimización contra Falsos Rechazos en Entornos Reales

**Fecha:** 21 de junio de 2026

### 1. Problema Técnico Detectado

Al testear fotos reales dentro del entorno del gimnasio, la aplicación sufría de una alta tasa de **"falsos rechazos"**. Imágenes anatómicamente correctas para un ojo humano eran descartadas por la IA debido a que elementos propios del entorno (espejos, oclusiones por discos de peso o encuadres de cámara recortados) bajaban transitoriamente la visibilidad absoluta de uno o dos puntos anatómicos secundarios.

### 2. Soluciones Implementadas (Refactorización del Core)

Se reemplazó la estricta validación de "mínimo absoluto" por un **algoritmo adaptativo por cobertura de puntos esenciales**:

* **Regulación de Confianza:** Se introdujo un control deslizable de `confidence` directamente en la UI. Al modificarse este slider, el umbral de aceptación del modelo se recalcula dinámicamente.


* **Estrategia Bifásica de Validación:**
* *Modo Flexible:* Ideal para fotos complejas en el gimnasio. Evalúa y da diagnóstico positivo si el tronco central (hombros y caderas) es plenamente visible y al menos 6 de los 8 puntos anatómicos principales pasan el umbral adaptativo.


* *Modo Estricto:* Diseñado para análisis técnicos de alta exigencia competitiva. Requiere obligatoriamente un mínimo de 7 de 8 puntos anatómicos limpios y una visibilidad estructural óptima.




* **Resultado en Producción:** La aplicación local fue re-verificada con éxito y responde de manera fluida y estable en el puerto local (`[http://127.0.0.1:7860](http://127.0.0.1:7860)`).
